**Archive notice.** This unrendered experimental draft records an older natural-logic assignment. It is not the current Assignments 5 and 6 notebook. The optional Stanza and external-data sections remain as design notes and are not part of the course runtime.

*Like Assignments 1-4, Assignments 5 and 6 are bundled together. You only need to do Tasks 1 and 2 for Assignment 5 and Task 3 for Assignment 6. **Task 4 is optional because of its difficulty.** I have left it here in case you'd like to take a crack at it. Whether or not you attempt Task 4, read through the entire notebook to see what it entails and how you might handle it.*

What should a natural language inference system do? It receives a _premise_ sentence and a _hypothesis_ sentence and must say whether we can infer the premise from the hypothesis. For instance, if (1) were our premise and (2) were our hypothesis, our system should respond _yes_.

1. Every firm polled saw costs grow more than expected, even after adjusting for inflation.
2. Every big company in the poll reported cost increases.

In MacCartney & Manning 2009 (henceforth, M&M), you read about one sort of system for doing this: a _natural logic_ system. This system works by (i) obtaining an _edit path_ from the premise and the hypothesis; (ii) mapping that edit path into an _inference path_; (iii) computing the _join_ of the inferences in this path to obtain a relation between the premise and the hypothesis; and (iv) checking whether there is a _forward entailment_ relation between the premise and the hypothesis.

The definition of the relations is given in Table 2 of the paper.

| Symbol           | Names                | Example                     | Set theoretic definition   |
|:----------------:|:--------------------:|:---------------------------:|:--------------------------:|
|$x \equiv y$      | equivalence          | couch $\equiv$ sofa         | $x = y$                    |
|$x \sqsubset y$   | forward entailment   | crow $\sqsubset$ bird       | $x \subset y$              |
|$x \sqsupset y$   | reverse entailment   | European $\sqsupset$ French | $x \supset y$              |
|$x \land y$       | negation             | human $\land$ nonhuman      | $x \cap y = \emptyset$ & $x \cup y = U$ |
|$x \mid y$        | alternation          | cat $\mid$ dog              | $x \cap y = \emptyset$ & $x \cup y \neq U$ |
|$x \smile y$      | cover                | animal $\smile$ nonhuman    | $x \cap y \neq \emptyset$ & $x \cup y = U$ |
|$x\;\#\;y$        | independence         | animal $\;\#\;$ nonhuman    | otherwise                  |

The table of joins is given below.

Tasks 1 and 2 develop the core system using the minimum-edit-distance paths from class and the default inference relations for atomic edits from Section 4 of M&M. Tasks 3 and 4 add lexical relations from WordNet and the environment-sensitive inferences from Section 5 of M&M. You will then test the system on the FraCaS dataset. 

## Task 1

*Lines:* 4

Define the `__add__` magic method for the `Inference` class below. This method should use `join_table` (defined above) to produce a set of `Inference`s by joining two inferences. For instance, animal $\sqsupset$ dog $\bowtie$ dog $\sqsupset$ greyhound = {animal $\sqsupset$ greyhound}. `__add__` must return a set because, as M&M discuss, joining two relations can produce an indeterminate result. (Their implementation treats all such indeterminate joins as #. We will not do that here, since it is useful to see _why_ they do so.)

Note that `__add__` should **not** be symmetric, since joins are not symmetric: animal $\sqsupset$ dog $\bowtie$ dog $\sqsubset$ mammal = {animal $\equiv$ mammal, animal $\sqsupset$ mammal, animal $\sqsubset$ mammal, animal $\smile$ mammal, animal $\#$ mammal}, but dog $\sqsubset$ mammal $\bowtie$ animal $\sqsupset$ dog isn't even a licit join.

Now test `Inference.__add__` using the `Editor` subclasses below.

These subclasses are initialized with input and/or output strings and a relation. For instance, "brindle" and "fawn" are two different colorings of greyhounds: no greyhound is both brindle and fawn, so they are in the | relation. Each is at least a [subsective modifier](https://en.wikipedia.org/wiki/Subsective_modifier) (all brindle greyhounds are greyhounds). If we delete one, we thus obtain a $\sqsubset$ relation; if we insert one, we get a $\sqsupset$ relation (the default relations for deletion and insertion discussed in M&M). 

Does the same inference hold for every adjective? No. Privative adjectives such as "fake" introduce a $|$ relation: fake greyhounds are not greyhounds (fake greyhound $|$ greyhound), and greyhounds are not fake greyhounds (greyhound $|$ fake greyhound).

Most substitutions involving "fake" will also yield a $|$ relation.

What about "virtuosic"? Insertion and deletion edits involving it should act like edits involving "brindle".

Use the following four sentences to write your tests. Apply an edit $e_1$ to sentence $s_i$ to obtain $e_1(s_i)$, then apply an edit $e_2$ to that result. Combine the inferences associated with $e_1$ and $e_2$ using `Inference.__add__`, and check the combined inference. Test at least one case whose result is a non-singleton set of inferences.

## Editor Libraries

How should we store editors and create defaults when one is missing? The `EditorLibrary` class stores and retrieves substitutions, deletions, and insertions. When a requested editor is absent, the library creates a default: substitutions get a `#` relation because we have no other default for them, while deletions and insertions get the default behavior defined by MacCartney & Manning.

## Task 2

*Lines:* 20

How can we avoid computing by hand the edits that convert one sentence into another? We will use a modified form of the `StringEdit` class developed in class. In particular, we need the edit paths that it produces.

First, we'll define a class for representing and manipulating edit paths. One important thing we want this class to do is to convert the edit path into a list of editors, for which we need to have a way to look up the editor for a given edit type and parameters given an `EditorLibrary`.

Now we'll adapt the class from lecture that computes edit distances, alignments, and edit paths between strings.

The original implementation indexed each edit into the source string so that we could identify a word's original position. That choice causes a problem here: after an insertion or deletion, the positions of later edits change. The implementation below corrects for this shift. Note that the order of edits matters for exactly this reason.

Implement the `__call__` method for the `NaturalLogic` class. It should take a premise sentence and a hypothesis sentence and produce the inference paths, computed from edit paths, that take you from the premise to the hypothesis. 

Each path should list the inferences produced by cumulatively composing the inference associated with each edit. It should **not** be a path of local inferences. That is, return the successive results of composition with `Inference.__add__`, not the uncomposed inference for each edit.

You will **not** be using `EditPath.__call__` in any way. That method is implemented to demonstrate how we should apply edit paths to strings. You should instead be using `EditPath.to_editors` to get the list of editors that result from the edit path, and then you should use those editors to compute the local inferences (again, the inferences that result from applying each editor in sequence).

Now implement tests using the four test sentences above. (Ignore my modified versions of these sentences.) For now, assume that the editor library contains the editors defined for Task 1. We don't need to specify insertions that result in $\sqsupset$ or deletions that result in $\sqsubset$, since `NaturalLogic.add_editor` adds those by default. In Task 3, we will expand the library using [WordNet](https://wordnet.princeton.edu/).

## Evaluating against FraCaS

How well does the resulting `NaturalLogic` implementation work? For Tasks 3 and 4, we will evaluate it on the [FraCaS textual inference test suite](https://nlp.stanford.edu/~wcmac/downloads/fracas.xml), which is shipped as XML.

The simple corpus reader below loads that XML.

This archived design note originally proposed an external lemmatization pipeline. Its dependency and network-triggering model download have been removed. The remaining note is non-executable history.

How do we turn an inference into a FraCaS prediction? Convert the inference produced by `__call__` into a "yes", "no", or "don't know" answer, ignoring items with any other label. This requires a mapping from inference types to answers. Then compute the accuracy, precision, recall, and F1 of the system.

Each metric can be defined in terms of the following counts:

1. The true positive count for class $c$: $$\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) = |\{i\;:\;y^\mathrm{test}_i = \hat{y}^\mathrm{test}_i = c\}|$$
2. The true negative count for class $c$: $$\mathrm{tn}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) = |\{i\;:\;y^\mathrm{test}_i \neq c \land \hat{y}^\mathrm{test}_i \neq c\}|$$
3. The false positive count for class $c$: $$\mathrm{fp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) = |\{i\;:\;y^\mathrm{test}_i \neq c \land \hat{y}^\mathrm{test}_i = c\}|$$
4. The false negative count for class $c$: $$\mathrm{fn}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) = |\{i\;:\;y^\mathrm{test}_i = c \land \hat{y}^\mathrm{test}_i \neq c\}|$$

Here the class is "yes", "no", or "unknown"; $y^\mathrm{test}_i$ is the true FraCaS label for item $i$, and $\hat{y}^\mathrm{test}_i$ is the system's prediction. Ignore cases whose class is not one of these three.

#### Accuracy

For what proportion of the test data $\{(x^\mathrm{test}_{1}, y^\mathrm{test}_1), ..., (x^\mathrm{test}_N, y^\mathrm{test}_N)\}$ does the predicted class $f(x^\mathrm{test}_i) = \hat{y}^\mathrm{test}_i$ match the item's ground-truth class?

$$\mathrm{accuracy}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}\right) = \frac{\sum_{c \in \mathcal{Y}}\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) + \mathrm{tn}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}{N}$$

[`sklearn.metrics`](https://scikit-learn.org/stable/modules/model_evaluation.html) provides an [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html#sklearn.metrics.accuracy_score) function, though it is also straightforward to compute the score directly.

#### Precision

For a particular class $c$, what proportion of the items predicted to have that class actually have it?

$$\mathrm{precision}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right) = \frac{\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}{\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) + \mathrm{fp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}$$

For aggregate precision across classes, we distinguish _micro-average_ from _macro-average_ precision.

$$\mathrm{microprecision}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}\right) = \frac{\sum_{c \in \mathcal{Y}} \mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}{\sum_{c \in \mathcal{Y}} \mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) + \mathrm{fp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}$$

$$\mathrm{macroprecision}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}\right) = \frac{1}{|\mathcal{Y}|}\sum_{c \in \mathcal{Y}} \mathrm{precision}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right)$$

#### Recall

For a particular class $c$, what proportion of the items with that class did the model predict correctly?

$$\mathrm{recall}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right) = \frac{\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}{\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) + \mathrm{fn}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}$$

Micro- and macro-average recall are defined in the same way as the corresponding precision averages.

#### F1

For a class $c$, $F_1$ is the [harmonic mean](https://en.wikipedia.org/wiki/Harmonic_mean) of precision and recall:

$$F_1\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right) = \frac{2}{\frac{1}{\mathrm{precision}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right)} + \frac{1}{\mathrm{recall}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right)}} = 2\frac{\mathrm{precision}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right)\;\cdot\;\mathrm{recall}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right)}{\mathrm{precision}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right) + \mathrm{recall}\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right)}$$ 

To define micro- and macro-average $F_1$, it is useful to reexpress the quantity in terms of counts.

$$F_1\left(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c\right) = \frac{2\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}{2\mathrm{tp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) + \mathrm{fp}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c) + \mathrm{fn}(\hat{\mathbf{y}}^\mathrm{test}_i, \mathbf{y}^\mathrm{test}, c)}$$

Micro- and macro-average $F_1$ can then be defined in parallel with the precision averages.

## Historical Task 3

This archived task proposed constructing an editor library from WordNet. The network-triggering corpus download has been removed, so the sketch below remains non-executable.

Now test the library with examples that require hypernymy, hyponymy, and antonymy.

Evaluate the new library on FraCaS by computing precision, recall, and F1 for the items labeled "yes", "no", or "don't know". This again requires a mapping from inference types to answers. 

Expect these numbers to be low. Even the apparently simple FraCaS cases are difficult for an edit-based system with a fairly extensive library. One reason is that the current system does not handle quantification or negation.

## Task 4

Update `NaturalLogic.__call__` to handle negation and the quantifiers discussed in Section 5 of M&M. Assume that "a" behaves as "some"; that "all" and "each" behave like "every"; that "not all" behaves like "not every"; and that "none" behaves like "no". This does not cover quantifiers such as "most" or "many"; do not try to infer projectivity signatures for those cases.

To do this, identify the first and second arguments of the quantifier. For instance, in (3), the first argument of "every" is "virtuosic synthesist" and the second is "loves a greyhound". (If you've taken semantics, you know I'm simplifying a little here.)

3. Every virtuosic synthesist loves some greyhound.

I have provided an implementation below. It works only for simple cases such as (3). Finding the arguments of quantifiers in general can be highly nontrivial, for reasons that a formal semantics course develops in detail.

Now note a complication: this implementation can produce a second argument for one quantifier that overlaps with the first argument of another.

This overlap requires an order for projecting inference relations through the quantifiers. In (3), for instance, should the system consider "every" before "a", or vice versa? The order will necessarily be heuristic. One option is the reverse of the linear (or "surface") order; another is an order based on depth in the tree.

Finally, apply the Task 3 evaluation to the new implementation of `NaturalLogic.__call__`.